# 🟠 Урок 16 (демо) — датасет → Colab → Gradio → публикация

Пройдём весь путь целиком на **нарисованном датасете фигур** (круг / квадрат / треугольник), чтобы всё гарантированно работало без единого своего фото. Потом заменишь генерацию на свои фото — остальной код не меняется.

**Что сделаем:** создадим датасет из картинок → обучим CNN → поднимем сайт Gradio → опубликуем его навсегда на Hugging Face Spaces.

> ⏱ На CPU обучение ~1–2 минуты. Если включишь GPU (Среда выполнения → Сменить среду → GPU) — секунды.

## Шаг 1 · Создаём датасет из картинок 🎨
Код сам рисует по 120 картинок каждой фигуры и раскладывает их по папкам — **одна папка = один класс**, ровно как будут лежать твои фото.

In [ ]:
import os, shutil, random, numpy as np
from PIL import Image, ImageDraw

random.seed(42); np.random.seed(42)

ROOT = "data"
if os.path.exists(ROOT): shutil.rmtree(ROOT)      # чистим, если запускаешь второй раз
classes = ["круг", "квадрат", "треугольник"]
PER = 120        # картинок на класс
SIZE = 96        # размер картинки

def draw_shape(kind):
    """Рисуем одну фигуру: случайный фон, цвет, размер, положение + лёгкий шум."""
    bg = tuple(np.random.randint(200, 256, 3))
    im = Image.new("RGB", (SIZE, SIZE), bg)
    d = ImageDraw.Draw(im)
    col = tuple(np.random.randint(0, 140, 3))
    x0, y0 = np.random.randint(5, 25), np.random.randint(5, 25)
    x1, y1 = SIZE - np.random.randint(5, 25), SIZE - np.random.randint(5, 25)
    if kind == "круг":         d.ellipse([x0, y0, x1, y1], fill=col)
    elif kind == "квадрат":    d.rectangle([x0, y0, x1, y1], fill=col)
    else:                       d.polygon([(SIZE//2, y0), (x0, y1), (x1, y1)], fill=col)
    arr = np.array(im).astype(np.int16) + np.random.randint(-12, 12, (SIZE, SIZE, 3))
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

for c in classes:
    os.makedirs(f"{ROOT}/{c}", exist_ok=True)
    for i in range(PER):
        draw_shape(c).save(f"{ROOT}/{c}/{c}_{i}.png")

print("Датасет готов:", {c: len(os.listdir(f"{ROOT}/{c}")) for c in classes})

## Шаг 2 · Посмотрим на данные глазами 👀
Всегда смотри на данные перед обучением — так замечаешь проблемы заранее.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 5, figsize=(10, 6))
for row, c in enumerate(classes):
    fnames = os.listdir(f"{ROOT}/{c}")[:5]
    for col, fn in enumerate(fnames):
        axes[row, col].imshow(Image.open(f"{ROOT}/{c}/{fn}"))
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(c, rotation=0, labelpad=40, fontsize=13)
plt.suptitle("По 5 примеров каждого класса")
plt.tight_layout(); plt.show()

## Шаг 3 · Загрузка + train/val split
Делим на обучающую (80%) и проверочную (20%) выборки — чтобы честно оценивать модель и видеть переобучение.

In [ ]:
import tensorflow as tf
from tensorflow import keras

train_ds = keras.utils.image_dataset_from_directory(
    ROOT, validation_split=0.2, subset="training", seed=42,
    image_size=(SIZE, SIZE), batch_size=16)
val_ds = keras.utils.image_dataset_from_directory(
    ROOT, validation_split=0.2, subset="validation", seed=42,
    image_size=(SIZE, SIZE), batch_size=16)

class_names = train_ds.class_names
print("Категории (запомни порядок!):", class_names)

## Шаг 4 · Строим и обучаем CNN 🧠
Маленькая свёрточная сеть: `Rescaling` нормализует пиксели в [0,1], два свёрточных блока ищут приметы, `Dense` принимает решение.

In [ ]:
model = keras.Sequential([
    keras.layers.Input((SIZE, SIZE, 3)),
    keras.layers.Rescaling(1./255),                      # пиксели 0..255 -> 0..1
    keras.layers.Conv2D(16, 3, activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 3, activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(len(class_names), activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history = model.fit(train_ds, validation_data=val_ds, epochs=8)

## Шаг 5 · Проверяем на свежих картинках ✅
Рисуем новые фигуры (модель их не видела) и смотрим, угадывает ли.

In [ ]:
plt.figure(figsize=(11, 3))
for k in range(5):
    kind = random.choice(classes)
    img = np.array(draw_shape(kind))[..., :3]
    x = tf.image.resize(img, (SIZE, SIZE))[None, ...]
    p = model.predict(x, verbose=0)[0]
    guess = class_names[int(np.argmax(p))]
    ok = "✅" if guess == kind else "❌"
    ax = plt.subplot(1, 5, k + 1); ax.imshow(img); ax.axis("off")
    ax.set_title(f"{ok} {guess}\n({int(100*np.max(p))}%)", fontsize=11)
plt.suptitle("Проверка на новых картинках"); plt.tight_layout(); plt.show()

## Шаг 6 · Временный сайт через Gradio 🌐
`share=True` даёт публичную ссылку `gradio.live` — работает, пока открыт ноутбук (~72 ч). Открой ссылку и загрузи картинку!

In [ ]:
!pip install gradio -q
import gradio as gr, numpy as np

def predict(img):
    img = np.array(img)[..., :3]                     # PNG с прозрачностью -> RGB
    x = tf.image.resize(img, (SIZE, SIZE))[None, ...]
    p = model.predict(x, verbose=0)[0]
    return {class_names[i]: float(p[i]) for i in range(len(class_names))}

gr.Interface(fn=predict, inputs=gr.Image(),
             outputs=gr.Label(num_top_classes=len(class_names)),
             title="Классификатор фигур",
             description="Загрузи картинку с кругом, квадратом или треугольником.").launch(share=True)

## Шаг 7 · Постоянная публикация на Hugging Face Spaces 🚀
`gradio.live` временный. Для постоянной ссылки нужен бесплатный хостинг **Hugging Face Spaces**. Туда заливаются 3 файла: `model.keras`, `app.py`, `requirements.txt`.

Ячейка ниже сохранит модель, сама напишет `app.py` (с твоими классами) и `requirements.txt` (с точной версией TensorFlow из этого Colab — это важно, иначе модель может не загрузиться) и скачает всё на компьютер.

In [ ]:
# 1) сохраняем обученную модель
model.save("model.keras")

# 2) пишем app.py (SIZE и class_names подставляются автоматически)
header  = "import gradio as gr, tensorflow as tf, numpy as np\n"
header += "from tensorflow import keras\n\n"
header += "SIZE = %d\n" % SIZE
header += "model = keras.models.load_model('model.keras')\n"
header += "class_names = %r\n\n" % (class_names,)
body = '''def predict(img):
    img = np.array(img)[..., :3]
    x = tf.image.resize(img, (SIZE, SIZE))[None, ...]
    p = model.predict(x)[0]
    return {class_names[i]: float(p[i]) for i in range(len(class_names))}

gr.Interface(fn=predict, inputs=gr.Image(),
             outputs=gr.Label(num_top_classes=len(class_names)),
             title="Классификатор фигур").launch()
'''
open("app.py", "w").write(header + body)

# 3) requirements.txt с ТОЧНОЙ версией TensorFlow из этого Colab
open("requirements.txt", "w").write(f"gradio\ntensorflow=={tf.__version__}\n")

print("Готовы 3 файла: model.keras, app.py, requirements.txt")
print("Версия TensorFlow вписана в requirements.txt:", tf.__version__)

# 4) скачиваем на компьютер
from google.colab import files
for fn in ["model.keras", "app.py", "requirements.txt"]:
    files.download(fn)

### Как залить на Spaces (веб-способ, проще для класса)
1. Зайди на **huggingface.co**, зарегистрируйся (бесплатно).
2. Открой **huggingface.co/new-space** → впиши имя → **Space SDK: Gradio** → Create Space.
3. На странице Space: вкладка **Files** → **Add file → Upload files** → загрузи все 3 файла (`model.keras`, `app.py`, `requirements.txt`) → **Commit**.
4. Space соберётся сам за минуту и появится постоянная ссылка вида `huggingface.co/spaces/твой_ник/имя`.

**Важно для класса:**
- Бесплатный тариф — CPU: предсказание идёт, просто чуть медленнее. Для демо ок.
- Space «засыпает» при простое и просыпается за пару секунд — но **ссылка постоянная** (в отличие от `gradio.live`).
- Порядок классов в `app.py` уже правильный — он взят из `class_names`.

## 🎯 Твои задания
- **Базовый:** доведи до постоянной ссылки на Spaces и открой её на телефоне.
- **Со ⭐:** добавь 4-ю фигуру (например, звезду) в `draw_shape` и переобучи.
- **Продвинутый:** замени генерацию фигур на СВОИ фото (папки `data/класс/…`) — весь остальной код останется тем же. Это и есть твой финальный проект.

### ✅ Проверь себя
1. Зачем нужен train/val split?
2. Что делает `share=True` и чем отличается от Spaces?
3. Почему в `requirements.txt` важно указать версию TensorFlow?

<details><summary>Ответы</summary>

1. Чтобы честно проверять модель на данных, которых она не видела, и замечать переобучение.
2. `share=True` даёт временную ссылку `gradio.live` (пока открыт ноутбук); Spaces — постоянный бесплатный хостинг.
3. Модель, сохранённая одной версией TF, может не загрузиться на другой — пин версии убирает эту ошибку.
</details>